# Day 2 — pandas Fundamentals
### DICT Data Analytics — Train the Trainer · Participant Notebook

**Python for Analytics: Series, DataFrames, filtering, and derived columns**

Aligned to the TESDA competency *Prepare data sets* — import data, apply basic
manipulation.

---

**How to use this notebook**

1. Run the **Setup** cell first. Run it again any time the session restarts.
2. Work through Sections 1, 2 and 3. Each has a check cell at the end.
3. The checker tells you *what* went wrong, not just pass or fail. Read it
   before you ask for help.
4. Run **Final Check** when all three sections pass.

Work in pairs. Explaining your reasoning to a partner is the closest thing to
teaching practice available inside a lab.

*Prepared by Nina Comia for the Department of Information and Communications
Technology.*


## Setup

In [ ]:
# ==========================================================================
#  SETUP  —  run this cell first, and run it again any time the session restarts
# ==========================================================================
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

CSV = "citizen_service_requests.csv"


def build_dataset(n_rows=1200, seed=2026):
    """Build the Day 2 practice file. Synthetic, but it behaves like a real extract."""
    rng = np.random.default_rng(seed)

    regions = ["NCR", "Region I", "Region III", "Region IV-A", "Region VI",
               "Region VII", "Region X", "Region XI", "CAR", "BARMM"]
    region_w = [.22, .07, .11, .14, .08, .10, .07, .08, .06, .07]
    offices = ["Civil Registry", "Business Permits", "Land Records",
               "Social Services", "Health Services", "Public Works"]
    services = ["Birth Certificate", "Business Permit Renewal", "Land Title Verification",
                "Senior Citizen ID", "Medical Assistance", "Road Repair Request",
                "Marriage Certificate", "Barangay Clearance"]
    channels = ["Walk-in", "Online Portal", "Mobile App", "Email", "Hotline"]
    channel_w = [.34, .28, .14, .13, .11]
    statuses = ["Resolved", "Closed", "Pending", "In Progress", "Escalated", "Withdrawn"]
    status_w = [.46, .17, .13, .12, .08, .04]

    request_id = [f"SR-2025-{i:05d}" for i in range(1, n_rows + 1)]
    region = rng.choice(regions, n_rows, p=region_w)
    office = rng.choice(offices, n_rows)
    service_type = rng.choice(services, n_rows)
    channel = rng.choice(channels, n_rows, p=channel_w)
    status = rng.choice(statuses, n_rows, p=status_w)

    start = pd.Timestamp("2025-01-01")
    offsets = rng.integers(0, 365, n_rows)
    filed = pd.to_datetime([start + pd.Timedelta(days=int(d)) for d in offsets])

    days = np.round(rng.gamma(2.0, 4.5, n_rows) + 1).astype(float)
    done = np.isin(status, ["Resolved", "Closed"])
    days[~done] = np.nan
    done_idx = np.flatnonzero(done)
    days[rng.choice(done_idx, 25, replace=False)] = 999

    fee = np.where(
        np.isin(service_type, ["Barangay Clearance", "Medical Assistance", "Road Repair Request"]),
        0.0,
        np.round(rng.choice([30, 50, 75, 100, 150, 200, 350, 500], n_rows) * 1.0, 2))
    fee[rng.choice(n_rows, 18, replace=False)] = np.nan

    sat = rng.choice([1, 2, 3, 4, 5], n_rows, p=[.07, .10, .21, .36, .26]).astype(float)
    sat[~done] = np.nan
    sat[rng.choice(done_idx, 90, replace=False)] = np.nan

    d = pd.DataFrame({
        "request_id": request_id, "date_filed": filed, "region": region,
        "office": office, "service_type": service_type, "channel": channel,
        "status": status, "days_to_resolve": days, "processing_fee": fee,
        "satisfaction_rating": sat,
    })

    ncr_idx = d.index[d["region"] == "NCR"].to_numpy()
    dirty = rng.choice(ncr_idx, 40, replace=False)
    variants = [" NCR", "NCR ", " ncr ", "ncr", "Ncr", "  NCR"]
    d.loc[dirty, "region"] = [variants[i % len(variants)] for i in range(40)]

    ds = d["date_filed"].dt.strftime("%Y-%m-%d")
    us = rng.choice(n_rows, 60, replace=False)
    ds.iloc[us] = d["date_filed"].iloc[us].dt.strftime("%m/%d/%Y")
    d["date_filed"] = ds
    return d


if not os.path.exists(CSV):
    build_dataset().to_csv(CSV, index=False)

df = pd.read_csv(CSV)
print("Loaded", CSV)
print("Shape:", df.shape)        # expect (1200, 10)


## Orientation — the four-command first look

Never analyze a file you have not looked at. Run these four before anything
else, every time, on every file. Read the output rather than scrolling past it.


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df["status"].value_counts()


## Checker

In [ ]:
# ==========================================================================
#  CHECKER  —  run this once. You do not need to read it, but you may.
# ==========================================================================
RESULTS = {}


def _fmt(v):
    if isinstance(v, pd.DataFrame):
        return f"a DataFrame with {len(v)} rows"
    if isinstance(v, pd.Series):
        return f"a Series with {len(v)} values"
    if isinstance(v, (list, tuple)):
        return f"{type(v).__name__} of {len(v)}: {v[:6]}{' ...' if len(v) > 6 else ''}"
    if isinstance(v, dict):
        return f"dict with {len(v)} entries"
    return repr(v)


def _record(section, task, ok, got, want, hint, reveal=True):
    RESULTS[(section, task)] = ok
    mark = "PASS" if ok else "FAIL"
    print(f"  [{mark}]  Task {task}")
    if not ok:
        print(f"         you gave : {_fmt(got)}")
        if want is not None and reveal:
            print(f"         expected : {_fmt(want)}")
        if hint:
            print(f"         hint     : {hint}")


def _get(name):
    return globals().get(name, None)


def check_section_1():
    print("Section 1 — Python fundamentals")
    f = _get("classify_speed")
    ok = callable(f)
    if ok:
        try:
            ok = (f(2) == "fast" and f(7) == "normal" and f(30) == "slow"
                  and f(np.nan) == "unknown")
        except Exception as e:
            ok = False
            print(f"         your function raised: {e}")
    if not callable(f):
        got = "classify_speed is not defined yet"
    else:
        try:
            got = f"classify_speed(2)={f(2)!r}, (7)={f(7)!r}, (30)={f(30)!r}, (nan)={f(np.nan)!r}"
        except Exception:
            got = "your function raised an error"
    _record(1, 1, ok, got, "fast / normal / slow / unknown",
            "Check the order: the pd.isna guard has to come before the comparisons.")

    got = _get("speed_labels")
    want = ["fast", "normal", "slow", "unknown"]
    _record(1, 2, list(got) == want if got is not None else False, got, want,
            "Call your function on 2, 7, 30 and np.nan, in that order.", reveal=False)

    got = _get("big_sales")
    want = [7100, 8800, 6250, 12000]
    _record(1, 3, list(got) == want if got is not None else False, got, want,
            "Keep the original order. Only values strictly above 5000.", reveal=False)

    got = _get("fee_buckets")
    ok = (isinstance(got, dict) and set(got) == set(FEES)
          and all(got[k] == ("free" if k == 0 else "paid") for k in got))
    _record(1, 4, ok, got, "a dict mapping every distinct fee to 'free' or 'paid'",
            "A dict comprehension over FEES. Note that repeated fees collapse into one key.")

    got = _get("n_free")
    want = sum(1 for v in FEES if v == 0)
    _record(1, 5, got == want, got, want,
            "Count the fees in FEES that equal 0 — not the entries in your dictionary.",
            reveal=False)
    return _summary(1)


def check_section_2():
    print("Section 2 — Filtering and sorting")
    _record(2, 1, _get("ncr_count") == 203, _get("ncr_count"), None,
            "An exact match on the region column: (df['region'] == 'NCR').sum()")

    got = _get("escalated_online")
    ok = isinstance(got, pd.DataFrame) and len(got) == 27
    _record(2, 2, ok, got, "a DataFrame, filtered on two conditions",
            "Two conditions, each in its own parentheses, joined with &.")

    got = _get("low_rated")
    ok = isinstance(got, pd.DataFrame) and len(got) == 109
    _record(2, 3, ok, got, "a DataFrame of the rows rated 1 or 2",
            "Use .isin([1, 2]) on satisfaction_rating.")

    got = _get("top10_slowest")
    ok = (isinstance(got, pd.DataFrame) and len(got) == 10
          and (got["days_to_resolve"] == 999).all())
    _record(2, 4, ok, got, "the 10 rows with the largest days_to_resolve",
            "Use df.nlargest(10, 'days_to_resolve'). Then look hard at what came back.")

    _record(2, 5, _get("missing_fee_count") == 18, _get("missing_fee_count"), None,
            "Blank, not zero. Use .isna().sum() — never == np.nan.")
    return _summary(2)


def check_section_3():
    print("Section 3 — Derived columns")
    ok = ("filed" in df.columns
          and pd.api.types.is_datetime64_any_dtype(df["filed"])
          and df["filed"].isna().sum() == 0)
    _record(3, 1, ok, df["filed"].dtype if "filed" in df.columns else "no column 'filed'",
            "a datetime64 column with no blanks",
            "pd.to_datetime(..., format='mixed'). Write it to a NEW column called 'filed'.")

    ok = ("month_name" in df.columns
          and (df["month_name"] == "January").sum() == 76)
    _record(3, 2, ok, "missing or wrong" if not ok else "ok",
            "month names taken from 'filed'",
            "df['filed'].dt.month_name()", reveal=False)

    ok = ("region_clean" in df.columns
          and (df["region_clean"] == "NCR").sum() == 243)
    _record(3, 3, ok,
            (df["region_clean"] == "NCR").sum() if "region_clean" in df.columns else "no column",
            None,
            "Strip the stray spaces and uppercase the text, in that order. "
            "Your count should go up, not stay at 203.")

    ok = ("speed_category" in df.columns
          and (df["speed_category"] == "unknown").sum() == 467
          and set(df["speed_category"].unique()) <= {"fast", "normal", "slow", "unknown"})
    _record(3, 4, ok, "missing or wrong" if not ok else "ok",
            "fast / normal / slow / unknown",
            "Apply classify_speed to days_to_resolve.")

    if "fee_bracket" in df.columns:
        b = df["fee_bracket"]
        ok = ((b[df["processing_fee"] == 0] == "none").all()
              and (b[df["processing_fee"].between(1, 100)] == "low").all()
              and (b[df["processing_fee"] > 100] == "high").all()
              and (b[df["processing_fee"].isna()] == "unknown").all())
        got = b.value_counts().to_dict()
    else:
        ok, got = False, "no column 'fee_bracket'"
    _record(3, 5, ok, got,
            "none / low / high, and 'unknown' for the 18 blanks",
            "Assign with df.loc[mask, 'fee_bracket'] = value. The 18 blanks are not free services.")

    got = _get("working_subset")
    need = {"request_id", "region_clean", "status", "speed_category", "fee_bracket"}
    ok = (isinstance(got, pd.DataFrame) and len(got) == 1200
          and need <= set(got.columns))
    _record(3, 6, ok, got,
            "a DataFrame of 1200 rows including the columns you would hand over",
            f"It must contain at least: {sorted(need)}")
    return _summary(3)


def _summary(section):
    items = {k: v for k, v in RESULTS.items() if k[0] == section}
    passed = sum(items.values())
    total = len(items)
    print(f"\n  Section {section}: {passed} of {total} tasks passing.")
    if passed < total:
        print("  Read the hints above, fix the task, and run this cell again.")
    print()
    return passed == total


def check_everything():
    print("=" * 74)
    print("FINAL CHECK  —  all three sections")
    print("=" * 74 + "\n")
    a = check_section_1()
    b = check_section_2()
    c = check_section_3()
    total = len(RESULTS)
    passed = sum(RESULTS.values())
    print("=" * 74)
    if a and b and c:
        print("""
   *  *  *   C O N G R A T U L A T I O N S   *  *  *

   All 16 tasks passed.

   You can now load a real file, interrogate it before trusting it,
   filter it without silently losing rows, and add columns safely.
   More to the point, you found a padded region value, a 999 sentinel
   and eighteen blanks that no error message would ever have told you
   about.

   That is the part worth teaching.

   Before you close this notebook:
     1. File > Download > .ipynb, and keep your copy.
     2. Download working_subset_day2.csv. Day 3 starts by loading it.
     3. Write down one thing from today you would teach first, and why.
""")
        print("=" * 74)
    else:
        failed = [f"Section {s}, Task {t}" for (s, t), ok in sorted(RESULTS.items()) if not ok]
        print(f"\n   {passed} of {total} tasks passing. Still to fix:\n")
        for f in failed:
            print(f"     - {f}")
        print("\n   Run the section check cells above for the detail on each one.")
        print("=" * 74)


print("Checker loaded. Use check_section_1(), check_section_2(), check_section_3(),")
print("and check_everything() at the end.")


---

# Section 1 — Python fundamentals

**Hands-On 1 · 60 minutes**

Functions, comprehensions, and what happens to a blank.

These are the pieces of Python that carry data work. Nothing here is new syntax — what is new is where each piece belongs.


In [ ]:
# Two small lists used in Section 1. Run this cell before the tasks.
SALES = [4200, 7100, 3900, 8800, 5000, 6250, 12000]
FEES  = [0, 50, 0, 150, 50, 0, 500, 100, 0]

print("SALES:", SALES)
print("FEES: ", FEES)


### Task 1.1 — Write classify_speed(days)

Turn a resolution time into a service level label.

- 3 days or fewer → `'fast'`
- 10 days or fewer → `'normal'`
- more than 10 → `'slow'`
- a missing value → `'unknown'`

Return the label. Do not print it.

In [ ]:
def classify_speed(days):
    """Turn a resolution time into a service level label."""
    # your code here
    pass


### Task 1.2 — Test it on four values

Build a list called `speed_labels` holding the result of calling your
function on `2`, `7`, `30` and `np.nan`, in that order.

In [ ]:
speed_labels = None   # replace with your answer

print(speed_labels)


### Task 1.3 — Keep the sales above 5000

Using a **list comprehension** (not a loop), build `big_sales` from `SALES`,
keeping only the values strictly greater than 5000. Keep the original order.

In [ ]:
big_sales = None   # replace with your answer

print(big_sales)


### Task 1.4 — Build a fee bucket dictionary

Using a **dictionary comprehension**, build `fee_buckets` from `FEES`.
Each distinct fee becomes a key. The value is `'free'` when the fee is 0 and
`'paid'` otherwise.

Then print how many entries the dictionary has, and compare that to how many
items are in `FEES`.

In [ ]:
fee_buckets = None   # replace with your answer

print(fee_buckets)
print("entries in the dictionary:", len(fee_buckets) if fee_buckets else 0)
print("items in FEES          :", len(FEES))


### Task 1.5 — Count the free services

Set `n_free` to the number of **fees in `FEES`** that are free.

Read that sentence carefully before you write the line.

In [ ]:
n_free = None   # replace with your answer

print("free fees:", n_free)


#### Check Section 1

In [ ]:
check_section_1()


---

# Section 2 — Filtering and sorting

**Hands-On 2 · 60 minutes**

Cut the service request file down to the rows you care about.

Write each number down on paper as you go, and next to it write how confident you are that it is right, from one to five. You will need both after lunch.


### Task 2.1 — Count the NCR requests

Set `ncr_count` to the number of rows where the `region` column is exactly
`'NCR'`. Print it.

In [ ]:
ncr_count = None   # replace with your answer

print("NCR requests:", ncr_count)


### Task 2.2 — Escalated requests from the Online Portal

Set `escalated_online` to the rows where `status` is `'Escalated'` **and**
`channel` is `'Online Portal'`. Print how many there are.

In [ ]:
escalated_online = None   # replace with your answer

print("rows:", len(escalated_online) if escalated_online is not None else 0)


### Task 2.3 — The lowest satisfaction ratings

Set `low_rated` to the rows with a satisfaction rating of 1 or 2.

Use `.isin()` rather than two conditions joined with `|`.

In [ ]:
low_rated = None   # replace with your answer

print("rows:", len(low_rated) if low_rated is not None else 0)


### Task 2.4 — The ten slowest requests

Set `top10_slowest` to the ten requests with the largest `days_to_resolve`.

Use `nlargest`. Then print the `days_to_resolve` column and look at it properly.

In [ ]:
top10_slowest = None   # replace with your answer

if top10_slowest is not None:
    print(top10_slowest[["request_id", "region", "status", "days_to_resolve"]])


### Task 2.5 — Count the missing fees

Set `missing_fee_count` to the number of rows with **no processing fee
recorded**. Blank, not zero.

In [ ]:
missing_fee_count = None   # replace with your answer

print("rows with no fee recorded:", missing_fee_count)
print("rows with a fee of zero  :", (df["processing_fee"] == 0).sum())


#### Check Section 2

In [ ]:
check_section_2()


---

# Section 3 — Derived columns

**Hands-On 3 · 60 minutes**

Build a subset you would be willing to hand to a colleague.

Everything you add from here is a claim about the data. Make sure you can defend each one.


### Task 3.1 — Parse the dates

Add a **new** column called `filed` holding `date_filed` parsed as real
dates. Keep the original `date_filed` column untouched.

This file stores dates in two different formats, so be explicit about it.

In [ ]:
# your code here


print(df["filed"].dtype)
print(df["filed"].isna().sum(), "unparsed")


### Task 3.2 — Add the month name

Add a `month_name` column holding the name of the month each request was
filed, taken from `filed`.

In [ ]:
# your code here


print(df["month_name"].value_counts().head())


### Task 3.3 — Clean the region column

Add a `region_clean` column with the region stripped of stray spaces and
converted to uppercase.

Then count the NCR rows again and compare it to what you wrote down this
morning.

In [ ]:
# your code here


print("exact match this morning:", (df["region"] == "NCR").sum())
print("after cleaning          :", (df["region_clean"] == "NCR").sum())


### Task 3.4 — Categorize the resolution speed

Add a `speed_category` column by applying `classify_speed` to
`days_to_resolve`.

Then look at how many rows come back `'unknown'`, and work out which statuses
produce them.

In [ ]:
# your code here


print(df["speed_category"].value_counts())
print()
print(df.loc[df["speed_category"] == "unknown", "status"].value_counts())


### Task 3.5 — Bracket the processing fee

Add a `fee_bracket` column using `.loc` for every assignment:

- `'none'` where the fee is 0
- `'low'` where the fee is 1 to 100
- `'high'` where the fee is above 100

Then deal with the rows that have no fee recorded. Think about what
`'none'` would be claiming about them before you decide.

In [ ]:
# your code here


print(df["fee_bracket"].value_counts(dropna=False))


### Task 3.6 — Export the working subset

Build `working_subset`: the columns you would actually be willing to hand to
a colleague. It must keep all 1,200 rows and must include at least
`request_id`, `region_clean`, `status`, `speed_category` and `fee_bracket`.

Write it to `working_subset_day2.csv`.

There is no single right column list. Be ready to defend yours.

In [ ]:
working_subset = None   # replace with your answer

# working_subset.to_csv("working_subset_day2.csv", index=False)
# print(working_subset.shape)


#### Check Section 3

In [ ]:
check_section_3()


---

# Final Check

Run this once all three sections are passing. It re-runs every task from the
top, so it is also the way to confirm nothing you changed later broke something
earlier.


In [ ]:
check_everything()
